In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------
# 한글 폰트 설정 (OS 자동 감지)
# ---------------------------------------
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')     # 윈도우
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스 / 구글 코랩
plt.rc('axes', unicode_minus=False)

In [2]:
import glob
import os


# 1. 상대 경로 설정 (현재 작업 폴더 내의 data 폴더)
folder_path = "./data"  # 또는 간단히 "data"

# 2. 해당 폴더의 모든 .csv 파일 경로 수집
file_list = glob.glob(os.path.join(folder_path, "*.csv"))

# 3. CSV 파일 일괄 읽기 (한글 깨짐 방지 cp949)
df_list = []
for file in file_list:
    temp_df = pd.read_csv(file, encoding="cp949")
    df_list.append(temp_df)

# 4. 하나의 데이터프레임으로 통합
df_src = pd.concat(df_list, ignore_index=True)
df_src.rename(
    columns={"전용면적(㎡)": "전용면적", "거래금액(만원)": "거래금액"},
    inplace=True,
)

df_src.info()
df_src.head(1)


<class 'pandas.DataFrame'>
RangeIndex: 219414 entries, 0 to 219413
Data columns (total 21 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   NO       219414 non-null  int64  
 1   시군구      219414 non-null  str    
 2   번지       219414 non-null  str    
 3   본번       219414 non-null  int64  
 4   부번       219414 non-null  int64  
 5   단지명      219414 non-null  str    
 6   전용면적     219414 non-null  float64
 7   계약년월     219414 non-null  int64  
 8   계약일      219414 non-null  int64  
 9   거래금액     219414 non-null  str    
 10  동        219414 non-null  str    
 11  층        219414 non-null  int64  
 12  매수자      219414 non-null  str    
 13  매도자      219414 non-null  str    
 14  건축년도     219414 non-null  int64  
 15  도로명      219414 non-null  str    
 16  해제사유발생일  219414 non-null  str    
 17  거래유형     219414 non-null  str    
 18  중개사소재지   219414 non-null  str    
 19  등기일자     219414 non-null  str    
 20  주택유형     83777 non-null   str    
dty

,NO,시군구,번지,본번,부번,단지명,전용면적,계약년월,계약일,거래금액,...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
0,1,서울특별시 중랑구 신내동,650,650,0,신내6대주,49.77,202312,31,"49,000",...,6,-,-,1996,신내로19길 42,-,중개거래,서울 중랑구,24.02.26,NaN


### 데이터 정제

In [6]:
df = df_src.copy()

# ----------------------------------------------------------------
# 1. 시군구 쪼개기
# ----------------------------------------------------------------
split_data = df['시군구'].str.split(' ', expand=True)

df['시'] = split_data[0]
df['구'] = split_data[1]

# ----------------------------------------------------------------
# 2. 매수자/매도자/ㅇㅇㅇㅇ 컬럼 삭제
# ----------------------------------------------------------------
del df['매수자']
del df['매도자']
del df['중개사소재지']

# ----------------------------------------------------------------
# 3. 계약연월일 일자 합치기
# ----------------------------------------------------------------
# 계약일자와 계약년월을 str 형태로 이어붙여 표기
date_str = df['계약년월'].astype(str) + df['계약일'].astype(str)
df['계약일자'] = pd.to_datetime(date_str)
# 계약년월을 YYYYMM(int) -> YYYY-MM 형식으로 변환
df['계약년월'] = df['계약일자'].dt.to_period('M')
# 계약년월 계약일 컬럼 삭제
df = df.drop(['계약일'], axis=1)

# ----------------------------------------------------------------
# 4. 전용면적을 평수로 전환하기
# ----------------------------------------------------------------
df['평수'] = (df['전용면적'] / 3.3058).round(1)

# ----------------------------------------------------------------
# 5. 거래금액을 int형으로 전환
# ----------------------------------------------------------------
df['거래금액'] = df['거래금액'].str.replace(',', '').astype(int)

# ----------------------------------------------------------------
# 6. 해제가 있는 행은 취소컬럼 생성
# ----------------------------------------------------------------
# 해제사유발생일 Nan 처리 및 dateTime 처리
df['해제사유발생일'] = df['해제사유발생일'].replace('-', np.nan)
df['해제사유발생일'] = pd.to_datetime(df['해제사유발생일'])
# 해제 사유 발생일 존재 시 '해제' 컬럼에 True False 표시
df['취소'] = df['해제사유발생일'].notna()

# ----------------------------------------------------------------
# 7. 등기여부 컬럼 추가, True / False 후 True만 집계 포함
# ----------------------------------------------------------------
# 등기일자 Nan 처리 및 dateTime 처리
df['등기일자'] = df['등기일자'].replace('-', np.nan)
df['등기일자'] = pd.to_datetime(df['등기일자'], format='%y.%m.%d', errors="coerce")

## 등기여부 라는 컬럼을 표시하고 True False 로 하고 True 인 애들만 집계에 포함하기 - 종호
# '등기일자'가 NaT가 아니면(값이 있으면) True, NaT이면 False
df["등기여부"] = df["등기일자"].notna()

# 결과 확인
df.info()
df.head(1)
# # 등기여부가 False인 행들만 표로 보기
# true_df = df[df["등기여부"] == True]

# # 상위 5개 출력
# true_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 219414 entries, 0 to 219413
Data columns (total 23 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   NO       219414 non-null  int64         
 1   시군구      219414 non-null  str           
 2   번지       219414 non-null  str           
 3   본번       219414 non-null  int64         
 4   부번       219414 non-null  int64         
 5   단지명      219414 non-null  str           
 6   전용면적     219414 non-null  float64       
 7   계약년월     219414 non-null  period[M]     
 8   거래금액     219414 non-null  int64         
 9   동        219414 non-null  str           
 10  층        219414 non-null  int64         
 11  건축년도     219414 non-null  int64         
 12  도로명      219414 non-null  str           
 13  해제사유발생일  11615 non-null   datetime64[us]
 14  거래유형     219414 non-null  str           
 15  등기일자     189066 non-null  datetime64[us]
 16  주택유형     83777 non-null   str           
 17  시        219414 non-n

,NO,시군구,번지,본번,부번,단지명,전용면적,계약년월,거래금액,동,...,해제사유발생일,거래유형,등기일자,주택유형,시,구,계약일자,평수,취소,등기여부
0,1,서울특별시 중랑구 신내동,650,650,0,신내6대주,49.77,2023-12,49000,608,...,NaT,중개거래,2024-02-26,NaN,서울특별시,중랑구,2023-12-31,15.1,False,True
